# 🏛️ oracc-parser — Quickstart

This notebook shows you the actual data available, how to parse it, and where everything lives on disk.

In [ ]:
!pip install display

## 🎯 Goals of this notebook

1. **Get the catalogue data** — download catalogues (metadata) from Zenodo
2. **Parse a project** — load word CSVs, inspect the project catalogue, and explore individual tablets
3. **Get flat DataFrames** — metadata, transliterations, and full combined tables ready for analysis
4. **Export** — save results as CSV or JSONL

## 1. Get the catalogue data (Zenodo)

Download project catalogues from Zenodo (doi: 10.5281/zenodo.20625379). These contain all the metadata stored per text by project.

In [ ]:
import os
from pathlib import Path
from oracc_parser.settings import DATA_DIR, OUTPUT_DIR, CACHE_DIR, JSONZIP_DIR

# Only catalogues are required upfront; word CSVs are fetched per-project on demand.
catalogues_ok = CATALOGUE_DIR.exists() and any(CATALOGUE_DIR.glob("*.csv"))

if not catalogues_ok:
    print("Downloading catalogues from Zenodo...")
    fetch_data()
else:
    print(f"✅ Data found in {zip_dir} ({len(list(zip_dir.glob('*.zip')))} projects)")

## 2. Parse a project

### How project data is organized on Zenodo

ORACC projects follow a two-level hierarchy: an **umbrella project** (e.g. `saao`) groups related **sub-projects** (e.g. `saao/saa01`, `saao/saa02`, …). On Zenodo, word CSVs are stored as one zip per umbrella — so `saao.zip` contains the data for all 22 State Archives of Assyria volumes.

When you request a sub-project for the first time, the package downloads the entire umbrella zip, extracts all sub-projects it contains, and caches them locally. Subsequent requests for any project in the same umbrella (e.g. switching from `saao/saa01` to `saao/saa02`) are instant — no further download needed.

A few projects have no umbrella (e.g. `babcity`, `balt`, and many more) and are stored as their own single-project zip.

For demonstration purposes, `babcity` was picked below as it is a small project with fast download.

**Change the variable `PROJECT` to any valid oracc project or subproject name to test with different projects!**

In [ ]:
from pathlib import Path
from oracc_parser import reference_data


# What directories does oracc-parser use?
print("📁 Directory layout:")
print(f"   Data dir:    {DATA_DIR}")
print(f"   JSON ZIPs:   {JSONZIP_DIR}")
print(f"   Output dir:  {OUTPUT_DIR}")
print(f"   Cache dir:   {CACHE_DIR}")

In [ ]:
# What ORACC projects exist in the world?
# We fetch the LIVE list from ORACC servers:
all_projects = reference_data.get_live_project_list()
print(f"🌍 ORACC has {len(all_projects)} public projects")
all_projects.head(10)

In [ ]:
### 2.1 Available projects and bulk download

# Load projects with empty text folders so we can exclude them
_meta = get_projects_metadata()
_empty_projects = set(
    _meta.loc[_meta['Is_Text_Folder_Empty'].str.lower() == 'yes', 'Project_Name']
)

# Check for downloaded ZIPs using configured path
zip_dir = jsonzip_dir()
print(zip_dir)
if zip_dir.exists():
    # Exclude the large Zenodo bundle ZIPs and projects with no texts
    _zenodo_zips = {'oracc_jsonzip_all.zip', 'oracc_html_translations.zip', 'plaides_scraped_data.zip'}
    project_zips = [
        f for f in zip_dir.glob('*.zip')
        if f.name not in _zenodo_zips
        and f.stem.replace('-', '/') not in _empty_projects
    ]
    downloaded = sorted([f.stem for f in project_zips])
    total_size_mb = sum(f.stat().st_size for f in project_zips) / (1024*1024)
    print(f"📦 {len(downloaded)} projects downloaded ({total_size_mb:.0f} MB total)")
    print()
    # Show first 20
    for i, name in enumerate(downloaded[:20]):
        size_mb = (zip_dir / f"{name}.zip").stat().st_size / (1024*1024)
        print(f"   {name:40s}  {size_mb:6.1f} MB")
    if len(downloaded) > 20:
        print(f"   ... and {len(downloaded)-20} more")
else:
    print("No downloaded projects found yet.")
    print("Run: python scripts/download_zenodo_data.py")


In [ ]:
import pandas as pd
from oracc_parser.settings import CATALOGUE_DIR, WORD_CSV_DIR

rows = []
for cat_file in sorted(CATALOGUE_DIR.glob("*.csv")):
    slug = cat_file.stem
    project = slug.replace("-", "/", 1)
    umbrella = slug.split("-")[0]
    n_texts = sum(1 for _ in open(cat_file, encoding="utf-8")) - 1  # fast line count
    downloaded = (WORD_CSV_DIR / slug).exists() and any((WORD_CSV_DIR / slug).glob("*.csv"))
    rows.append({"project": project, "umbrella": umbrella, "texts": n_texts, "downloaded": "✓" if downloaded else ""})

# Check for downloaded ZIPs using configured path
zip_dir = jsonzip_dir()
print(zip_dir)
if zip_dir.exists():
    # Exclude the large Zenodo bundle ZIPs and projects with no texts
    _zenodo_zips = {'oracc_jsonzip_all.zip', 'oracc_html_translations.zip', 'plaides_scraped_data.zip'}
    project_zips = [
        f for f in zip_dir.glob('*.zip')
        if f.name not in _zenodo_zips
        and f.stem.replace('-', '/') not in _empty_projects
    ]
    downloaded = sorted([f.stem for f in project_zips])
    total_size_mb = sum(f.stat().st_size for f in project_zips) / (1024*1024)
    print(f"📦 {len(downloaded)} projects downloaded ({total_size_mb:.0f} MB total)")
    print()
    # Show first 20
    for i, name in enumerate(downloaded[:20]):
        size_mb = (zip_dir / f"{name}.zip").stat().st_size / (1024*1024)
        print(f"   {name:40s}  {size_mb:6.1f} MB")
    if len(downloaded) > 20:
        print(f"   ... and {len(downloaded)-20} more")
else:
    print("No downloaded projects found yet.")
    print("Run: python scripts/download_zenodo_data.py")


In [ ]:
# Compare: which projects are NOT yet downloaded?
from oracc_parser.utils.paths import get_projects_metadata
# Load projects with empty text folders so we can exclude them
_meta = get_projects_metadata()
_empty_projects = set(
    _meta.loc[_meta['Is_Text_Folder_Empty'].str.lower() == 'yes', 'Project_Name']
)
# Check for downloaded ZIPs using configured path
zip_dir = jsonzip_dir()

if zip_dir.exists() and 'all_projects' in dir():
    _zenodo_zips = {'oracc_jsonzip_all.zip', 'oracc_html_translations.zip', 'plaides_scraped_data.zip'}

    downloaded_names = {
        f.stem.replace('-', '/') for f in zip_dir.glob('*.zip')
        if f.name not in _zenodo_zips
        and f.stem.replace('-', '/') not in _empty_projects
    }
    live_names = set(all_projects['project'].tolist()) if 'project' in all_projects.columns else set()

    # Exclude empty-text projects from the live list too
    live_names -= _empty_projects

    not_downloaded = sorted(live_names - downloaded_names)

    print(f"📊 Download status:")
    print(f"   ✅ {len(downloaded_names)} projects downloaded")
    print(f"   ❌ {len(not_downloaded)} projects NOT yet downloaded")
    print(f"   🌍 {len(live_names)} total projects on ORACC (with texts)")
    print()
    if not_downloaded:
        print("Projects not yet downloaded:")
        for name in not_downloaded:
            print(f"   • {name}")
        print()
        print("To download one: oracc-parser download --project <name>")
        if len(not_downloaded) > 10:
            print("NOTE: If there are many new projects or a large mismatch, it means ORACC has updated significant projects.")
            print("   We recommend downloading the new projects directly using the function below.")
    else:
        print("🎉 All available projects are downloaded!")


In [ ]:
# ⬇️ Download Data
# You can download any project that was not yet downloaded by us, by name.
from oracc_parser.download.oracc_download import download_projects

projects_to_download = [
   "saao/saa01"
]


if projects_to_download:
    download_projects(projects_to_download)


## 3. Parse a project

Pick any project from the lists above. We'll parse a small sample to see the output.

In [ ]:
from collections import defaultdict
from tqdm import tqdm
from oracc_parser.download.fetch_data import extract_project_csvs

# Group catalogue entries by umbrella
umbrellas = defaultdict(list)
for slug in sorted(f.stem for f in CATALOGUE_DIR.glob("*.csv")):
    umbrellas[slug.split("-")[0]].append(slug)

# Only download umbrella groups that are not fully on disk yet
to_download = [
    (umbrella, slugs) for umbrella, slugs in sorted(umbrellas.items())
    if not all(
        (WORD_CSV_DIR / s).exists() and any((WORD_CSV_DIR / s).glob("*.csv"))
        for s in slugs
    )
]

if not to_download:
    print("All projects already downloaded.")
else:
    print(f"Downloading {len(to_download)} umbrella group(s) covering "
          f"{sum(len(s) for _, s in to_download)} project(s)...")
    skipped = []
    for umbrella, slugs in tqdm(to_download, unit="group"):
        missing = next(
            s for s in slugs
            if not (WORD_CSV_DIR / s).exists() or not any((WORD_CSV_DIR / s).glob("*.csv"))
        )
        try:
            extract_project_csvs(missing.replace("-", "/", 1))
        except (FileNotFoundError, ValueError) as e:
            skipped.append((umbrella, str(e).split(":")[0]))
    if skipped:
        print(f"\nSkipped {len(skipped)} group(s) with no downloadable word CSVs:")
        for name, reason in skipped:
            print(f"  {name}: {reason}")
    print("Done. Re-run the cell above to see updated download status.")

## 3. Get flat DataFrames (for analysis & export)

In [ ]:
# Look at one tablet up close
tablet = records[0]

print(f"=== {tablet.metadata.identifier} ===")
print(f"   Project:     {PROJECT}")
print(f"   Text ID:     {tablet.metadata.id_text}")
print(f"   Genre:       {tablet.metadata.genre}")

geo = tablet.metadata.geographical_information
if geo:
    print(f"   Provenance:  {geo.city.city_name}")
    print(f"   Pleiades ID: {geo.city.city_plaides_id}")

chrono = tablet.metadata.chronological_information
if chrono:
    print(f"   Period:      {chrono.tablet_period.period_name if chrono.tablet_period else ''}")
    print(f"   Years:       {chrono.start_year} to {chrono.end_year}")

print(f"   Words:       {len(tablet.content.words)}")

In [ ]:
# The text in different representations
tablet = records[0]
c = tablet.content

print("TRANSLITERATION:")
print(f"{c.transliterated_str_representation.text[:200] if c.transliterated_str_representation else '(none)'}")
print()
print("NORMALIZATION:")
print(f"{c.normalized_str_representation.text[:200] if c.normalized_str_representation else '(none)'}")
print()
print("UNICODE CUNEIFORM:")
print(f"{c.unicode_str_representation.text[:200] if c.unicode_str_representation else '(none)'}")
print()
print("ENGLISH TRANSLATION:")
print(f"{c.english_translation[:200] if c.english_translation else '(none)'}")

## 5. Get flat DataFrames (for analysis & export)

Each function returns a clean pandas DataFrame — no nesting, no Pydantic.

In [ ]:
from oracc_parser import get_metadata_table, get_transliterations, get_full_flat_table

# Metadata table
print("=== METADATA ===")
meta = get_metadata_table(records)
display(meta)

In [ ]:
# Transliterations
print("=== TRANSLITERATIONS ===")
trans = get_transliterations(records)
display(trans)

In [ ]:
from oracc_parser import get_metadata_table, get_transliterations, get_full_flat_table

# Full flat table — everything in one DataFrame
flat = get_full_flat_table(records)
print(f"Full table: {flat.shape[0]} rows × {flat.shape[1]} columns")
print(f"Columns: {list(flat.columns)}")
display(flat)

## 4. Export & see where output goes

The table above can be saved locally in as a JSONL or CSV format. The output directory of this package is under `enriched_data`. If you want to change it to another directory, change the value of the variable `out_dir`.

In [ ]:
from oracc_parser.settings import OUTPUT_DIR

out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = out_dir / f"{PROJECT.replace('/', '_')}.jsonl"
csv_path   = out_dir / f"{PROJECT.replace('/', '_')}.csv"

flat.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
flat.to_csv(csv_path, index=False)

print(f"✓ Exported to:")
print(f"   JSONL: {jsonl_path}  ({jsonl_path.stat().st_size / 1024:.1f} KB)")
print(f"   CSV:   {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")
print()
print(f"📁 Output directory: {out_dir}")
for f in sorted(out_dir.iterdir()):
    print(f"   {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

## 7. Reference data (bundled with the package)

These datasets ship with oracc-parser — no download needed.

In [ ]:
from oracc_parser import reference_data

print("Bundled reference data:")
datasets = {
    "Provenance (cities → Pleiades IDs)": reference_data.get_provenance(),
    "Period mapping (period → years)": reference_data.get_period_mapping(),
    "Sign readings (8900+ signs)": reference_data.get_sign_list(),
    "POS tags": reference_data.get_pos_tags(),
    "Languages": reference_data.get_languages(),
    "ORACC projects metadata": reference_data.get_projects_metadata(),
}
for name, df in datasets.items():
    print(f"   {name:45s}  {len(df):>6} rows × {len(df.columns)} cols")

In [ ]:
# Explore any of them
print("=== Provenance ===")
display(reference_data.get_provenance().head(10))

print("\n=== Period mapping ===")
display(reference_data.get_period_mapping())

## What's next?

- **Clean the data:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **Explore reference data:** See notebook `02_reference_data.ipynb` for a catalog of available projects, pos tags, and languages.